In [0]:
df = spark.table("e_commerce_2026.bronze.e_commerce_bronze")
display(df)

In [0]:
#Conferindo numero de linhas
df.count()

In [0]:
#conferindo o tipo de dados
df.printSchema()

In [0]:
#Criando uma "view" temporária na memória da sessão, usada como fonte pro MERGE INTO.
df_incremental = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv("/Volumes/e_commerce_2026/bronze/volume/pedidos_lote_incremental.csv")
)

df_incremental.createOrReplaceTempView("pedidos_incremental_view")

In [0]:
display(df_incremental)

In [0]:
df_incremental.count()

In [0]:
df_incremental.printSchema()

In [0]:
%sql

--USING ... ON destino.order_id = origem.order_id → compara cada linha do lote incremental com a tabela Gold pela chave order_id
--WHEN MATCHED → se o order_id já existe na tabela → UPDATE SET * atualiza todas as colunas daquela linha com os valores novos (ex: o --status muda de "Pendente" pra "Enviado")
--WHEN NOT MATCHED → se o order_id não existe → INSERT * insere a linha inteira como pedido novo




MERGE INTO e_commerce_2026.bronze.e_commerce_bronze AS destino
USING pedidos_incremental_view AS origem
ON destino.order_id = origem.order_id
WHEN MATCHED THEN
  UPDATE SET *
WHEN NOT MATCHED THEN
  INSERT *

In [0]:
%sql
SELECT COUNT(*) FROM e_commerce_2026.bronze.e_commerce_bronze;

In [0]:
%sql
DESCRIBE HISTORY e_commerce_2026.bronze.e_commerce_bronze;

In [0]:
%sql
SELECT order_id, COUNT(*) AS qtd
FROM e_commerce_2026.bronze.e_commerce_bronze
GROUP BY order_id
HAVING COUNT(*) > 1
ORDER BY qtd DESC;

In [0]:
%sql
CREATE TABLE e_commerce_2026.silver.pedidos_silver AS
SELECT * FROM e_commerce_2026.bronze.e_commerce_bronze;

In [0]:
%sql
SELECT COUNT(*) FROM e_commerce_2026.silver.pedidos_silver;